In [1]:
import pickle
import numpy as np
import pandas as pd
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn import set_config

# scikit-learn 모델 HTML 출력 비활성화
set_config(display="text")

In [2]:
data = pd.read_csv(
    "../risk_data/Sleep Health and Lifestyle Dataset.csv"
)

display(data.head())

,Patient_ID,Age,Gender,Occupation,Marital_Status,Physical_Activity_Minutes,Screen_Time_Hours,Daily_Steps,Water_Intake_Liters,Caffeine_Intake_mg,...,Depression_Score,Workload_Score,Snoring_Frequency,Respiratory_Disturbance_Index,Leg_Movement_Index,Sleep_Risk_Index,Lifestyle_Risk_Index,Physiological_Risk_Index,Mental_Health_Risk_Index,Sleep_Disorder
0,1,49,Female,Engineer,Married,38.7,3.2,3666,2.90,148,...,6,7,8,33.6,12.4,4.48,2.46,15.59,6.0,Sleep Apnea
1,2,40,Female,Student,Married,55.9,8.2,6239,2.65,138,...,2,3,2,2.8,14.1,2.22,4.52,6.34,6.5,No Disorder
2,3,42,Female,Doctor,Married,53.3,3.9,6932,2.67,195,...,1,4,2,4.1,5.5,2.07,3.00,6.56,6.5,No Disorder
3,4,51,Male,Engineer,Married,47.6,7.1,6036,1.88,183,...,2,6,2,7.6,8.6,1.90,4.60,5.66,6.5,No Disorder
4,5,44,Female,Analyst,Single,87.1,6.3,11413,2.49,205,...,4,5,2,0.0,7.4,2.15,3.74,5.08,6.0,No Disorder


In [3]:
required_columns = [
    "Age",
    "Gender",
    "BMI",
    "Sleep_Duration",
    "Daytime_Sleepiness",
    "Caffeine_Intake_mg",
    "Physical_Activity_Minutes",
    "Screen_Time_Hours",
    "Night_Awakenings",
    "Smoking_Status",
    "Alcohol_Consumption",
    "Sleep_Efficiency"
]

missing_columns = [
    column
    for column in required_columns
    if column not in data.columns
]

if missing_columns:
    raise ValueError(
        f"누락된 컬럼: {missing_columns}"
    )

In [4]:
target = "Sleep_Efficiency"


numeric_features = [
    "Age",
    "BMI",
    "Sleep_Duration",
    "Daytime_Sleepiness",
    "Caffeine_Intake_mg",
    "Physical_Activity_Minutes",
    "Screen_Time_Hours",
    "Night_Awakenings"
]


categorical_features = [
    "Gender",
    "Smoking_Status",
    "Alcohol_Consumption"
]


input_features = (
    numeric_features
    + categorical_features
)


print("=== Final Input Features ===")

for index, feature in enumerate(
    input_features,
    start=1
):
    print(f"{index}. {feature}")

print("\n총 입력 Feature:", len(input_features))

=== Final Input Features ===
1. Age
2. BMI
3. Sleep_Duration
4. Daytime_Sleepiness
5. Caffeine_Intake_mg
6. Physical_Activity_Minutes
7. Screen_Time_Hours
8. Night_Awakenings
9. Gender
10. Smoking_Status
11. Alcohol_Consumption

총 입력 Feature: 11


In [5]:
X = data[input_features].copy()
y = data[target].copy()


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (24000, 11)
X_test : (6000, 11)
y_train: (24000,)
y_test : (6000,)


In [6]:
outlier_bounds = {
    "Caffeine_Intake_mg": (0, 395),
    "Physical_Activity_Minutes": (0, 104.4)
}


for feature, (lower, upper) in outlier_bounds.items():

    X_train[feature] = (
        X_train[feature]
        .clip(lower, upper)
    )

    X_test[feature] = (
        X_test[feature]
        .clip(lower, upper)
    )


print("이상치 처리 완료")

이상치 처리 완료


In [7]:
scaler = StandardScaler()

X_train_numeric = scaler.fit_transform(
    X_train[numeric_features]
)

X_test_numeric = scaler.transform(
    X_test[numeric_features]
)


encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_categorical = encoder.fit_transform(
    X_train[categorical_features]
)

X_test_categorical = encoder.transform(
    X_test[categorical_features]
)


X_train_processed = np.hstack([
    X_train_numeric,
    X_train_categorical
])

X_test_processed = np.hstack([
    X_test_numeric,
    X_test_categorical
])


print(
    "Train Shape:",
    X_train_processed.shape
)

print(
    "Test Shape :",
    X_test_processed.shape
)

Train Shape: (24000, 14)
Test Shape : (6000, 14)


In [8]:
model = GradientBoostingRegressor(
    random_state=42
)


model.fit(
    X_train_processed,
    y_train
)

GradientBoostingRegressor(random_state=42)

In [9]:
train_pred = model.predict(
    X_train_processed
)

test_pred = model.predict(
    X_test_processed
)


train_mae = mean_absolute_error(
    y_train,
    train_pred
)

test_mae = mean_absolute_error(
    y_test,
    test_pred
)

train_rmse = np.sqrt(
    mean_squared_error(
        y_train,
        train_pred
    )
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        test_pred
    )
)

train_r2 = r2_score(
    y_train,
    train_pred
)

test_r2 = r2_score(
    y_test,
    test_pred
)


metrics = {
    "Train_MAE": train_mae,
    "Test_MAE": test_mae,
    "Train_RMSE": train_rmse,
    "Test_RMSE": test_rmse,
    "Train_R2": train_r2,
    "Test_R2": test_r2
}


print("=== Final Model Performance ===")

for metric, value in metrics.items():
    print(
        f"{metric}: {value:.4f}"
    )

=== Final Model Performance ===
Train_MAE: 3.7049
Test_MAE: 3.7659
Train_RMSE: 4.6160
Test_RMSE: 4.6931
Train_R2: 0.6436
Test_R2: 0.6329


In [10]:
sleep_durations = [
    2, 4, 6, 7, 8, 9, 10
]


test_cases = []


for duration in sleep_durations:

    test_cases.append({
        "Age": 32,
        "Gender": "Female",
        "BMI": 22.5,
        "Sleep_Duration": duration,
        "Daytime_Sleepiness": 5,
        "Caffeine_Intake_mg": 95,
        "Physical_Activity_Minutes": 45,
        "Screen_Time_Hours": 5,
        "Night_Awakenings": 2,
        "Smoking_Status": "No",
        "Alcohol_Consumption": "No"
    })


duration_input = pd.DataFrame(
    test_cases
)


# 이상치 처리
for feature, (lower, upper) in outlier_bounds.items():

    duration_input[feature] = (
        duration_input[feature]
        .clip(lower, upper)
    )


# 전처리
duration_numeric = scaler.transform(
    duration_input[numeric_features]
)

duration_categorical = encoder.transform(
    duration_input[categorical_features]
)

duration_processed = np.hstack([
    duration_numeric,
    duration_categorical
])


# 예측
duration_predictions = model.predict(
    duration_processed
)


duration_result = pd.DataFrame({
    "Sleep_Duration": sleep_durations,
    "Predicted_Sleep_Efficiency":
        duration_predictions
})


display(
    duration_result.round(4)
)

,Sleep_Duration,Predicted_Sleep_Efficiency
0,2,79.5633
1,4,79.7568
2,6,81.8214
3,7,83.5928
4,8,84.3423
5,9,84.4977
6,10,82.9377


In [13]:
model_package = {
    "model": model,

    "scaler": scaler,
    "encoder": encoder,

    "numeric_features":
        numeric_features,

    "categorical_features":
        categorical_features,

    "input_features":
        input_features,

    "outlier_bounds":
        outlier_bounds,

    "target":
        target,

    "model_name":
        "GradientBoostingRegressor",

    "sklearn_version":
        sklearn.__version__,

    "metrics": {
        "train_mae": train_mae,
        "test_mae": test_mae,
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "train_r2": train_r2,
        "test_r2": test_r2
    }
}


MODEL_PATH = (
    "../risk_data/efficiency_final_model.pkl"
)


with open(
    MODEL_PATH,
    "wb"
) as file:

    pickle.dump(
        model_package,
        file
    )


print(
    f"모델 저장 완료: {MODEL_PATH}"
)

모델 저장 완료: ../risk_data/efficiency_final_model.pkl


In [14]:
with open(
    MODEL_PATH,
    "rb"
) as file:

    loaded_package = pickle.load(
        file
    )


print(
    "Model:",
    loaded_package["model_name"]
)

print(
    "Target:",
    loaded_package["target"]
)

print(
    "Input Features:",
    loaded_package["input_features"]
)

print(
    "Input Feature Count:",
    len(
        loaded_package[
            "input_features"
        ]
    )
)

Model: GradientBoostingRegressor
Target: Sleep_Efficiency
Input Features: ['Age', 'BMI', 'Sleep_Duration', 'Daytime_Sleepiness', 'Caffeine_Intake_mg', 'Physical_Activity_Minutes', 'Screen_Time_Hours', 'Night_Awakenings', 'Gender', 'Smoking_Status', 'Alcohol_Consumption']
Input Feature Count: 11


In [15]:
test_user = pd.DataFrame([{
    "Age": 32,
    "Gender": "Female",
    "BMI": 22.5,
    "Sleep_Duration": 7,
    "Daytime_Sleepiness": 5,
    "Caffeine_Intake_mg": 95,
    "Physical_Activity_Minutes": 45,
    "Screen_Time_Hours": 5,
    "Night_Awakenings": 2,
    "Smoking_Status": "No",
    "Alcohol_Consumption": "No"
}])


loaded_model = (
    loaded_package["model"]
)

loaded_scaler = (
    loaded_package["scaler"]
)

loaded_encoder = (
    loaded_package["encoder"]
)

loaded_numeric_features = (
    loaded_package[
        "numeric_features"
    ]
)

loaded_categorical_features = (
    loaded_package[
        "categorical_features"
    ]
)

loaded_outlier_bounds = (
    loaded_package[
        "outlier_bounds"
    ]
)


# 이상치 처리
for feature, (lower, upper) in (
    loaded_outlier_bounds.items()
):

    test_user[feature] = (
        test_user[feature]
        .clip(lower, upper)
    )


# 전처리
test_numeric = (
    loaded_scaler.transform(
        test_user[
            loaded_numeric_features
        ]
    )
)

test_categorical = (
    loaded_encoder.transform(
        test_user[
            loaded_categorical_features
        ]
    )
)


test_processed = np.hstack([
    test_numeric,
    test_categorical
])


# 예측
prediction = (
    loaded_model.predict(
        test_processed
    )[0]
)


print(
    "Predicted Sleep Efficiency:"
    f" {prediction:.2f}%"
)

Predicted Sleep Efficiency: 83.59%
